# Phase 0 — Qwen3-VL-2B trên Kaggle T4
Tải bản `phase0_bundle.zip` mới lên một private Kaggle Dataset, gắn vào notebook qua Add Input. Chạy trong phiên mới để tránh dùng mã cũ còn trong Python cache. Notebook bootstrap mã và 20 ảnh gốc; ảnh lớn được thu nhỏ tối đa 1280 pixel mỗi cạnh trước suy luận và kích thước xử lý được ghi trong kết quả.


In [ ]:
from pathlib import Path
import sys, zipfile
INPUT_ROOT = Path('/kaggle/input/vlm-phase0-bundle')  # sửa theo slug Kaggle thực tế
archive = INPUT_ROOT / 'phase0_bundle.zip'
if archive.is_file():
    BUNDLE_ROOT = Path('/kaggle/working/vlm_phase0_bundle')
    BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as bundle_zip:
        for item in bundle_zip.infolist():
            target = (BUNDLE_ROOT / item.filename).resolve()
            assert target.is_relative_to(BUNDLE_ROOT.resolve()), 'Unsafe archive path'
        bundle_zip.extractall(BUNDLE_ROOT)
else:
    BUNDLE_ROOT = INPUT_ROOT
assert (BUNDLE_ROOT / 'vlm_phase0' / 'run_phase0.py').is_file(), 'Add the supplied bundle as Kaggle Dataset Input'
assert (BUNDLE_ROOT / 'manifest.jsonl').is_file(), 'Missing 20-image manifest.jsonl'
sys.path.insert(0, str(BUNDLE_ROOT))
from vlm_phase0.qwen import QwenBaseline
assert QwenBaseline.resize_longest_edge == 1280, 'Upload the updated phase0_bundle.zip and restart the session'
print('Image resize longest edge:', QwenBaseline.resize_longest_edge)
MODEL_REVISION = '89644892e4d85e24eaac8bacfd4f463576704203'  # same commit as first Kaggle run


## Dependencies
Dùng PyTorch CUDA có sẵn của Kaggle; ghi chính xác versions sau cài. Nếu pip báo xung đột, dừng và ghi lại thay vì đổi model.


In [ ]:
%pip install -q 'transformers>=4.57,<5' 'accelerate>=1,<2' 'huggingface-hub>=0.34,<1' 'Pillow>=10,<13'


In [ ]:
import subprocess
OUTPUT = Path('/kaggle/working/vlm_phase0/reports')
OUTPUT.mkdir(parents=True, exist_ok=True)
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, check=True)
(OUTPUT / 'pip_freeze.txt').write_text(freeze.stdout, encoding='utf-8')
preflight = subprocess.run([sys.executable, '-m', 'vlm_phase0.run_phase0', '--preflight-only', '--output-dir', str(OUTPUT)], cwd=str(BUNDLE_ROOT), text=True, capture_output=True)
print(preflight.stdout, preflight.stderr, 'exit=', preflight.returncode)
print((OUTPUT / 'hardware_report.json').read_text(encoding='utf-8'))


## 20 ảnh thật
Sau khi xác nhận T4/driver/CUDA/quota, chạy lại toàn bộ baseline với chính sách resize đã ghi. `smoke_review.csv` cần người chấm xem ảnh và điền verdict; chữ nhỏ có thể mất sau khi thu nhỏ. Notebook không tự chấm correctness.


In [ ]:
assert preflight.returncode == 0
run = subprocess.run([sys.executable, '-m', 'vlm_phase0.run_phase0', '--manifest', str(BUNDLE_ROOT / 'manifest.jsonl'), '--dataset-root', str(BUNDLE_ROOT), '--output-dir', str(OUTPUT), '--model-revision', MODEL_REVISION], cwd=str(BUNDLE_ROOT), text=True, capture_output=True)
print(run.stdout, run.stderr, 'exit=', run.returncode)
report = OUTPUT / 'smoke_report.json'
if report.exists(): print(report.read_text(encoding='utf-8'))
assert run.returncode == 0, 'Read hardware report and per-sample errors; retain notebook output for diagnosis'
